# **Time Series Coding Case study**

This notebook exemplifies appliance energy usage as a time series case study to evaluate and develop multiple forecasting approaches.

In [8]:
import pandas as pd
from pathlib import Path

# Part 1: **Data Retrieval and EDA**

**Data Retrieval**

Here, we retrieve the Appliances Energy Prediction dataset, convert the timestamp to a datetime index, remove the non-informative random variables, convert the remaining columns to numeric format, check for missing values, and resample the data from 10-minute to hourly resolution.



In [9]:
# Build IPYNB structure

target = "Appliances"
raw_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00374/energydata_complete.csv"

data_dir = Path("data")
processed_path = data_dir / "processed" / "appliance_hourly.csv"
processed_path.parent.mkdir(parents=True, exist_ok=True)


In [10]:
# Helper function


# Load dataset
def load_data(raw_url):
    """Download the raw 10-minute Appliances Energy Prediction CSV and parse its timestamp."""
    df = pd.read_csv(raw_url)

    # Format date and time structure
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()

    # drop random noise columns
    df = df.drop(columns=["rv1", "rv2"], errors="ignore")

    # Convert columns to numeric, coercing invalid values to NaN
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


# clean and resample loaded dataset


def clean(df: pd.DataFrame, freq: str = "h"):
    """Drop missing targets, resample to `freq`, and interpolate any small resulting gaps."""

    # Remove row with missing Target
    out = df.dropna(subset=[target]).copy()

    # Group into fixed 1 hour intevrals, clean, and fill-in time periods
    resampled = out.resample(freq).mean()
    resampled = resampled.interpolate("time")
    resampled = resampled.dropna()

    return resampled

**Load** dataset using function defined above:
> `load_data(raw_url)`

In [11]:
data = load_data(raw_url)
print("Data shape:", data.shape)
print("Date range:", data.index.min(), "to", data.index.max())
print("Inferred sampling interval:", data.index.to_series().diff().median())
data.head()

Data shape: (19735, 26)
Date range: 2016-01-11 17:00:00 to 2016-05-27 18:00:00
Inferred sampling interval: 0 days 00:10:00


,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,...,T8,RH_8,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint
date,,,,,,,,,,,,,,,,,,,,,
2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,45.566667,...,18.2,48.900000,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3
2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,45.992500,...,18.2,48.863333,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2
2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,45.890000,...,18.2,48.730000,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1
2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,45.723333,...,18.1,48.590000,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0
2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,45.530000,...,18.1,48.590000,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9


In [12]:
# Check for missing values
missing = data.isna().sum()
print(missing[missing > 0] if missing.sum() > 0 else "There are no missing values.")

There are no missing values.


From `data` above, we can hence see that the dataset contains 19,735 rows of observation and 26 columns recorded at 10 minutes intervals.


**Clean and Resample** dataset to hourly data  using function defined above:
> `clean(df: pd.DataFrame, freq: str = "h")`

In [13]:
df = clean(data, freq="h")

print("Hourly shape:", df.shape)
print("Missing values after resample + interpolate:", df.isna().sum().sum())

df.to_csv(processed_path)
print("Saved to", processed_path.resolve())

df.head()

Hourly shape: (3290, 26)
Missing values after resample + interpolate: 0
Saved to /content/data/processed/appliance_hourly.csv


,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,...,T8,RH_8,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint
date,,,,,,,,,,,,,,,,,,,,,
2016-01-11 17:00:00,55.000000,35.000000,19.890000,46.502778,19.200000,44.626528,19.790000,44.897778,18.932778,45.738750,...,18.150000,48.710556,17.016667,45.446667,6.308333,733.750000,92.000000,6.166667,53.416667,5.050000
2016-01-11 18:00:00,176.666667,51.666667,19.897778,45.879028,19.268889,44.438889,19.770000,44.863333,18.908333,46.066667,...,18.094444,48.597222,16.981667,45.290000,5.941667,734.266667,91.583333,5.416667,40.000000,4.658333
2016-01-11 19:00:00,173.333333,25.000000,20.495556,52.805556,19.925556,46.061667,20.052222,47.227361,18.969444,47.815556,...,18.156111,49.213333,16.902222,45.311389,6.000000,734.791667,89.750000,6.000000,40.000000,4.391667
2016-01-11 20:00:00,125.000000,35.000000,20.961111,48.453333,20.251111,45.632639,20.213889,47.268889,19.190833,49.227917,...,18.773333,50.195556,16.890000,45.118889,6.000000,735.283333,87.583333,6.000000,40.000000,4.016667
2016-01-11 21:00:00,103.333333,23.333333,21.311667,45.768333,20.587778,44.961111,20.373333,46.164444,19.425556,47.918889,...,19.153333,49.542222,16.890000,44.807778,5.833333,735.566667,87.416667,6.000000,40.000000,3.816667


In [14]:
app = df[target]
app.describe()

,Appliances
count,3290.000000
mean,97.779129
std,81.213695
min,28.333333
25%,50.000000
50%,63.333333
75%,110.000000
max,608.333333


The dataset resampled hourly now contains 3,290 rows from 19,735 rows at 10-minute intervals and still with no missing values after interpolation.

